# Faza I – Paraprocesimi i të dhënave (Eurostat)

**Dataset:** `estat_tour_occ_nim_en.csv`  
**Qëllimi:** analizimi i strukturës dhe cilësisë së të dhënave, pastrimi, trajtimi i vlerave që mungojnë, mostrimi, agregimi, krijimi/përzgjedhja e veçorive, diskretizimi, binarizimi, PCA dhe transformimi.

## Si të përdoret
1. Vendoseni këtë notebook në të njëjtën dosje me skedarin CSV.
2. Instaloni bibliotekat nëse nevojiten: `pip install pandas numpy matplotlib scikit-learn`.
3. Ekzekutoni qelizat nga lart-poshtë.
4. Rezultatet eksportohen në dosjen `output_phase1`.

## 1. Importimi i bibliotekave dhe ngarkimi i datasetit

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore", category=FutureWarning)

INPUT_FILE = Path("estat_tour_occ_nim_en.csv")
if not INPUT_FILE.exists():
    fallback = Path("estat_tour_occ_nim_en.csv")
    if fallback.exists():
        INPUT_FILE = fallback
    else:
        raise FileNotFoundError("Vendoseni CSV-në në të njëjtën dosje me notebook-un ose ndryshoni INPUT_FILE.")

OUTPUT_DIR = Path("output_phase1")
OUTPUT_DIR.mkdir(exist_ok=True)

df_original = pd.read_csv(INPUT_FILE, low_memory=False)
df = df_original.copy()

print("Forma:", df.shape)
display(df.head())
print(df.columns.tolist())

## 2. Tipet e të dhënave dhe fjalori i variablave

In [ ]:
df.info()

type_report = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "unique_values": [df[c].nunique(dropna=True) for c in df.columns],
    "missing_values": [int(df[c].isna().sum()) for c in df.columns],
    "missing_percent": [(df[c].isna().mean() * 100).round(2) for c in df.columns],
})
display(type_report)
type_report.to_csv(OUTPUT_DIR / "01_data_types_report.csv", index=False)

column_descriptions = {
    "DATAFLOW": "Identifikuesi i rrjedhës së të dhënave",
    "LAST UPDATE": "Koha e përditësimit të datasetit",
    "freq": "Frekuenca (M = mujore)",
    "c_resid": "Kategoria e rezidentëve",
    "unit": "Njësia/treguesi statistikor",
    "nace_r2": "Kategoria e aktivitetit ekonomik",
    "geo": "Shteti/territori",
    "TIME_PERIOD": "Periudha kohore (vit-muaj)",
    "OBS_VALUE": "Vlera numerike e vëzhgimit",
    "OBS_FLAG": "Shenja/flag-u i vëzhgimit",
    "CONF_STATUS": "Statusi i konfidencialitetit",
}
variable_report = pd.DataFrame({
    "column": df.columns,
    "description": [column_descriptions.get(c, "Përshkrim për t'u verifikuar") for c in df.columns],
    "actual_dtype": [str(df[c].dtype) for c in df.columns],
    "unique_values": [df[c].nunique(dropna=True) for c in df.columns],
})
display(variable_report)
variable_report.to_csv(OUTPUT_DIR / "02_variable_dictionary.csv", index=False)

## 3. Cilësia e të dhënave: vlera që mungojnë, dublikata dhe vlera ekstreme

In [ ]:
missing_report = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})
display(missing_report)
missing_report.to_csv(OUTPUT_DIR / "03_missing_values_report.csv")

full_duplicates = int(df.duplicated().sum())
key_columns = ["freq", "c_resid", "unit", "nace_r2", "geo", "TIME_PERIOD"]
key_duplicate_count = int(df.duplicated(subset=key_columns).sum())
print("Dublikatat e plota:", full_duplicates)
print("Dublikatat sipas dimensioneve kryesore:", key_duplicate_count)

for col in ["freq", "c_resid", "unit", "nace_r2", "geo", "OBS_FLAG", "CONF_STATUS"]:
    if col in df.columns:
        print(f"\nVlerat në {col}:")
        display(df[col].value_counts(dropna=False).head(15))

display(df["OBS_VALUE"].describe())

flag_missing_report = pd.crosstab(
    df["OBS_FLAG"].fillna("(flag mungon)"),
    df["OBS_VALUE"].isna(),
    rownames=["OBS_FLAG"],
    colnames=["OBS_VALUE mungon"],
    margins=True,
)
display(flag_missing_report)
flag_missing_report.to_csv(OUTPUT_DIR / "05_missing_obs_by_flag.csv")

obs_non_missing = df["OBS_VALUE"].dropna()
q1, q3 = obs_non_missing.quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_outliers = df[df["OBS_VALUE"].notna() & ((df["OBS_VALUE"] < lower_bound) | (df["OBS_VALUE"] > upper_bound))]
print("Vlera jashtë kufijve IQR (vetëm sinjal diagnostikues):", len(iqr_outliers))
iqr_outliers.head(1000).to_csv(OUTPUT_DIR / "06_potential_outliers_sample.csv", index=False)

## 4. Pastrimi i të dhënave dhe strategjia për vlerat që mungojnë

In [ ]:
df_clean = df_original.copy()

for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = df_clean[col].map(lambda x: x.strip() if isinstance(x, str) else x)

df_clean["OBS_VALUE"] = pd.to_numeric(df_clean["OBS_VALUE"], errors="coerce")
df_clean["period_date"] = pd.to_datetime(
    df_clean["TIME_PERIOD"].astype(str) + "-01",
    format="%Y-%m-%d",
    errors="coerce",
)

df_analysis = df_clean[df_clean["OBS_VALUE"].notna()].copy()

constant_columns = [c for c in df_analysis.columns if df_analysis[c].nunique(dropna=False) <= 1]
df_analysis_reduced = df_analysis.drop(columns=constant_columns, errors="ignore")

print("Forma fillestare:", df_original.shape)
print("Forma pas pastrimit:", df_clean.shape)
print("Vlera OBS_VALUE që mungojnë:", df_clean["OBS_VALUE"].isna().sum())
print("Rreshta me OBS_VALUE të disponueshme:", len(df_analysis))
print("Kolona konstante në kopjen analitike:", constant_columns)

df_clean.to_csv(OUTPUT_DIR / "cleaned_dataset_with_missing_values.csv", index=False)
df_analysis.to_csv(OUTPUT_DIR / "analysis_dataset_non_missing.csv", index=False)

## 5. Mostrimi

In [ ]:
sample_size = min(5000, len(df_analysis))
df_sample = df_analysis.sample(n=sample_size, random_state=42)
df_sample.to_csv(OUTPUT_DIR / "random_sample.csv", index=False)
print("Madhësia e mostrës:", len(df_sample))
display(df_sample.head())

## 6. Agregimi vjetor

In [ ]:
df_for_annual = df_analysis.copy()
df_for_annual["year"] = df_for_annual["period_date"].dt.year
annual_keys = ["geo", "c_resid", "unit", "nace_r2"]

nr_df = df_for_annual[df_for_annual["unit"] == "NR"]
other_df = df_for_annual[df_for_annual["unit"] != "NR"]

annual_parts = []
if not nr_df.empty:
    annual_nr = nr_df.groupby(annual_keys + ["year"], dropna=False).agg(
        annual_value=("OBS_VALUE", "sum"),
        available_months=("OBS_VALUE", "count")
    ).reset_index()
    annual_nr["aggregation_method"] = "sum_monthly_NR"
    annual_parts.append(annual_nr)

if not other_df.empty:
    annual_other = other_df.groupby(annual_keys + ["year"], dropna=False).agg(
        annual_value=("OBS_VALUE", "mean"),
        available_months=("OBS_VALUE", "count")
    ).reset_index()
    annual_other["aggregation_method"] = "mean_monthly_non_NR"
    annual_parts.append(annual_other)

annual_summary = pd.concat(annual_parts, ignore_index=True) if annual_parts else pd.DataFrame()
if not annual_summary.empty:
    annual_summary["expected_months"] = np.where(
        annual_summary["year"] == df_for_annual["year"].max(),
        df_for_annual.groupby("year")["TIME_PERIOD"].nunique().get(df_for_annual["year"].max(), 12),
        12
    )
    annual_summary["months_coverage_percent"] = (
        annual_summary["available_months"] / annual_summary["expected_months"].replace(0, np.nan) * 100
    ).round(2)
    annual_summary.to_csv(OUTPUT_DIR / "annual_aggregated_summary.csv", index=False)
display(annual_summary.head())